# Vilier Kaggle ASR workflow

Notebook này tách riêng pha ASR để chạy trên cloud GPU như Kaggle. Luồng khuyến nghị:

1. Local chạy các bước trước ASR để tạo `outputs/<audio_id>/vad_audio`, `vad.json`, `manifest.timeline.json`.
2. Local chạy mode `local_prepare` để đóng gói bundle và optional upload thành Kaggle Dataset.
3. Kaggle GPU chạy mode `kaggle_run`: clone repo, unzip input bundle, chạy `tools/run_asr_bundle.py`.
4. Local chạy mode `local_merge` để nhập `transcript.json` về `outputs/<audio_id>/`, rồi tiếp tục state labeling/review local.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

# Chọn một trong ba mode: local_prepare, kaggle_run, local_merge
MODE = "local_prepare"

# Local config
LOCAL_PROJECT_ROOT = Path.cwd()
AUDIO_ID = "podcast_single_30s"
LOCAL_OUTPUT_DIR = LOCAL_PROJECT_ROOT / "outputs" / AUDIO_ID
LOCAL_PACKAGE_DIR = LOCAL_PROJECT_ROOT / "kaggle_asr_packages"

# Kaggle config
# Đổi URL này sang repo thật của bạn trước khi chạy trên Kaggle.
PROJECT_GIT_URL = "https://github.com/ngocbao220/vilier.git"
PROJECT_REF = "main"
KAGGLE_DATASET_DIR = Path("/kaggle/input/vilier-asr-bundle")
KAGGLE_WORK_DIR = Path("/kaggle/working")
KAGGLE_REPO_DIR = KAGGLE_WORK_DIR / "vilier"
KAGGLE_BUNDLE_DIR = KAGGLE_WORK_DIR / "asr_bundle" / AUDIO_ID
KAGGLE_ASR_OUTPUT_DIR = KAGGLE_WORK_DIR / "asr_result"
ASR_DEVICE = "0"  # Kaggle GPU. Dùng "cpu" nếu notebook không bật GPU.
KAGGLE_ASR_DRY_RUN = True  # Smoke test trước; đổi False để chạy PhoWhisper thật.

# Optional local Kaggle Dataset upload. Cần kaggle.json credentials.
RUN_KAGGLE_UPLOAD = False
KAGGLE_DATASET_SLUG = "ngocbaotrinhtuan/vilier-asr-bundle"

# Local merge config: trỏ tới file zip tải từ Kaggle output.
DOWNLOADED_ASR_ZIP = LOCAL_PACKAGE_DIR / f"{AUDIO_ID}_asr_result.zip"

print("mode", MODE)
print("audio_id", AUDIO_ID)


## 1. Local prepare

Chạy cell này trên máy local sau khi `outputs/<audio_id>/` đã có `vad_audio/`, `vad.json`, và `manifest.timeline.json`.

In [ ]:
if MODE == "local_prepare":
    required = [LOCAL_OUTPUT_DIR / "manifest.timeline.json", LOCAL_OUTPUT_DIR / "vad.json", LOCAL_OUTPUT_DIR / "vad_audio"]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing prepared ASR inputs: " + ", ".join(missing))

    LOCAL_PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
    bundle_zip = LOCAL_PACKAGE_DIR / f"{AUDIO_ID}_asr_bundle.zip"
    include_names = {"manifest.timeline.json", "vad.json"}
    with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for item in LOCAL_OUTPUT_DIR.rglob("*"):
            if item.is_dir():
                continue
            rel = item.relative_to(LOCAL_OUTPUT_DIR)
            if rel.parts[0] == "vad_audio" or rel.name in include_names:
                zf.write(item, rel.as_posix())

    print("wrote", bundle_zip)
    print("size_mb", round(bundle_zip.stat().st_size / 1024 / 1024, 2))
    print("Next: upload this zip as a Kaggle Dataset, or set RUN_KAGGLE_UPLOAD=True below.")


In [ ]:
if MODE == "local_prepare" and RUN_KAGGLE_UPLOAD:
    bundle_zip = LOCAL_PACKAGE_DIR / f"{AUDIO_ID}_asr_bundle.zip"
    if not bundle_zip.exists():
        raise FileNotFoundError(bundle_zip)
    dataset_dir = LOCAL_PACKAGE_DIR / "kaggle_dataset"
    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)
    dataset_dir.mkdir(parents=True)
    shutil.copy2(bundle_zip, dataset_dir / bundle_zip.name)
    metadata = {
        "title": f"Vilier ASR bundle {AUDIO_ID}",
        "id": KAGGLE_DATASET_SLUG,
        "licenses": [{"name": "CC0-1.0"}],
    }
    (dataset_dir / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    cmd = ["kaggle", "datasets", "create", "-p", str(dataset_dir), "--dir-mode", "zip"]
    print("running", " ".join(cmd))
    subprocess.run(cmd, check=True)


## 2. Kaggle run

Chạy các cell dưới trên Kaggle Notebook có bật GPU. Add Kaggle Dataset chứa file `<audio_id>_asr_bundle.zip` vào notebook trước khi chạy.

In [ ]:
if MODE == "kaggle_run":
    print(subprocess.check_output(["nvidia-smi"], text=True))
    if KAGGLE_REPO_DIR.exists():
        shutil.rmtree(KAGGLE_REPO_DIR)
    subprocess.run(["git", "clone", PROJECT_GIT_URL, str(KAGGLE_REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(KAGGLE_REPO_DIR), "checkout", PROJECT_REF], check=True)
    sys.path.insert(0, str(KAGGLE_REPO_DIR))
    print("repo", KAGGLE_REPO_DIR)


In [ ]:
if MODE == "kaggle_run":
    # Kaggle thường đã có torch GPU. Cài các runtime deps còn thiếu cho ASR-only.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "soundfile", "librosa", "pyyaml"],
        check=True,
    )
    bundle_candidates = sorted(KAGGLE_DATASET_DIR.rglob(f"{AUDIO_ID}_asr_bundle.zip"))
    if not bundle_candidates:
        bundle_candidates = sorted(KAGGLE_DATASET_DIR.rglob("*.zip"))
    if not bundle_candidates:
        raise FileNotFoundError(f"No bundle zip found under {KAGGLE_DATASET_DIR}")
    bundle_zip = bundle_candidates[0]
    if KAGGLE_BUNDLE_DIR.exists():
        shutil.rmtree(KAGGLE_BUNDLE_DIR)
    KAGGLE_BUNDLE_DIR.mkdir(parents=True)
    with zipfile.ZipFile(bundle_zip) as zf:
        zf.extractall(KAGGLE_BUNDLE_DIR)
    print("bundle", bundle_zip)
    print("extracted", KAGGLE_BUNDLE_DIR)


In [ ]:
if MODE == "kaggle_run":
    if KAGGLE_ASR_OUTPUT_DIR.exists():
        shutil.rmtree(KAGGLE_ASR_OUTPUT_DIR)
    KAGGLE_ASR_OUTPUT_DIR.mkdir(parents=True)
    cmd = [
        sys.executable,
        str(KAGGLE_REPO_DIR / "tools" / "run_asr_bundle.py"),
        "--config",
        str(KAGGLE_REPO_DIR / "config.json"),
        "--bundle-dir",
        str(KAGGLE_BUNDLE_DIR),
        "--output-dir",
        str(KAGGLE_ASR_OUTPUT_DIR),
        "--device",
        ASR_DEVICE,
    ]
    if KAGGLE_ASR_DRY_RUN:
        cmd.append("--dry-run")
    print("running", " ".join(map(str, cmd)))
    env = os.environ.copy()
    env["PYTHONPATH"] = str(KAGGLE_REPO_DIR)
    subprocess.run(cmd, check=True, env=env)
    result_zip = KAGGLE_WORK_DIR / f"{AUDIO_ID}_asr_result.zip"
    with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for item in KAGGLE_ASR_OUTPUT_DIR.rglob("*"):
            if item.is_file():
                zf.write(item, item.relative_to(KAGGLE_ASR_OUTPUT_DIR).as_posix())
    print("download from Kaggle output:", result_zip)


## 3. Local merge

Sau khi tải file `<audio_id>_asr_result.zip` từ Kaggle về local, đặt path vào `DOWNLOADED_ASR_ZIP` rồi chạy cell này.

In [ ]:
if MODE == "local_merge":
    if not DOWNLOADED_ASR_ZIP.exists():
        raise FileNotFoundError(DOWNLOADED_ASR_ZIP)
    if not LOCAL_OUTPUT_DIR.exists():
        raise FileNotFoundError(LOCAL_OUTPUT_DIR)
    extract_dir = LOCAL_PACKAGE_DIR / f"{AUDIO_ID}_asr_result"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    with zipfile.ZipFile(DOWNLOADED_ASR_ZIP) as zf:
        zf.extractall(extract_dir)
    transcript_src = extract_dir / "transcript.json"
    if not transcript_src.exists():
        raise FileNotFoundError(transcript_src)
    transcript = json.loads(transcript_src.read_text(encoding="utf-8"))
    transcript_dst = LOCAL_OUTPUT_DIR / "transcript.json"
    transcript_dst.write_text(json.dumps(transcript, ensure_ascii=False, indent=2), encoding="utf-8")

    manifest_path = LOCAL_OUTPUT_DIR / "manifest.timeline.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        manifest["transcript"] = transcript
        manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print("merged", len(transcript), "records into", transcript_dst)


In [ ]:
if MODE == "local_merge":
    # Optional: chạy state labeling local từ transcript đã tải về.
    import sys
    sys.path.insert(0, str(LOCAL_PROJECT_ROOT))
    from pipeline.cli import load_config, state_dir_for_audio
    from pipeline.labeling import label_transcripts, load_labeling_runner, resolve_state_dir, write_state_outputs
    from pipeline.asr import write_transcript_json

    RUN_LOCAL_STATE_LABELING = False
    if RUN_LOCAL_STATE_LABELING:
        config = load_config(LOCAL_PROJECT_ROOT / "config.json")
        runner = load_labeling_runner(config, dry_run=False)
        if runner is None:
            raise ValueError("state_labeling is disabled")
        labeled = label_transcripts(transcript, runner)
        state_dir = state_dir_for_audio(LOCAL_OUTPUT_DIR, resolve_state_dir(config))
        summary = write_state_outputs(AUDIO_ID, LOCAL_OUTPUT_DIR, state_dir, labeled)
        write_transcript_json(LOCAL_OUTPUT_DIR / "transcript.json", labeled)
        manifest_path = LOCAL_OUTPUT_DIR / "manifest.timeline.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        manifest["transcript"] = labeled
        manifest["state_labeling"] = summary
        manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
        print("state labeled", summary)
